# Full Evaluation Notebook

Notebook này phục vụ phần đánh giá trong báo cáo: nhiều metric, 5-fold cross validation, bảng/biểu đồ main results, ablation study, error analysis và timing. Pipeline hiện tại dùng mô hình pre-trained + luật, không có bước huấn luyện supervised trong repo, nên cross validation ở đây là cross-validation để ước lượng độ ổn định của cấu hình pipeline trên các fold dữ liệu.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import sys
import tempfile
import time
import traceback

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'dataset' / 'image'
SCENE_CONFIG = PROJECT_ROOT / 'configs' / 'scene_config.example.json'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
EVAL_DIR = PROJECT_ROOT / 'outputs' / 'full_evaluation' / RUN_ID
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Evaluation dir:', EVAL_DIR)

## Install/Import Evaluation Libraries

In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        accuracy_score,
        average_precision_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_recall_curve,
        precision_score,
        recall_score,
        roc_auc_score,
        roc_curve,
    )
    from sklearn.model_selection import StratifiedKFold
except ImportError:
    print('Missing eval libraries. In Colab run: !pip install scikit-learn pandas matplotlib')
    raise

## Settings

In [ ]:
N_SPLITS = 5
MAX_IMAGES = None
TASK_FILTER = None
RUN_VLM_RECHECK = False
QWEN_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VLM_MAX_NEW_TOKENS = 512

BASE_THRESHOLD = 0.25
BASE_TEXT_THRESHOLD = 0.20

print({
    'n_splits': N_SPLITS,
    'max_images': MAX_IMAGES,
    'task_filter': TASK_FILTER,
    'run_vlm_recheck': RUN_VLM_RECHECK,
})

## Dataset Labels

Binary label lấy từ folder: `violate` = 1, `not_violate` = 0. Violation type label chỉ dùng để phân tích phụ vì dataset hiện tại không có annotation chi tiết đáng tin cậy cho từng loại vi phạm ở mọi ảnh.

In [ ]:
def infer_binary_label(image_path: Path) -> int:
    parts = set(image_path.relative_to(DATASET_DIR).parts)
    if 'not_violate' in parts:
        return 0
    if 'violate' in parts:
        return 1
    return -1

def infer_task(image_path: Path) -> str:
    rel_parts = image_path.relative_to(DATASET_DIR).parts
    return rel_parts[0] if rel_parts else 'unknown'

image_paths = sorted(
    p for p in DATASET_DIR.rglob('*')
    if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}
)

records = []
for path in image_paths:
    label = infer_binary_label(path)
    task = infer_task(path)
    if label < 0:
        continue
    if TASK_FILTER and task != TASK_FILTER:
        continue
    records.append({
        'image_path': path,
        'relative_path': str(path.relative_to(PROJECT_ROOT)),
        'task': task,
        'y_true': label,
        'label_name': 'violate' if label else 'not_violate',
    })

if MAX_IMAGES is not None:
    records = records[:MAX_IMAGES]

df_data = pd.DataFrame([{k: v for k, v in row.items() if k != 'image_path'} for row in records])
display(df_data.groupby(['task', 'label_name']).size().reset_index(name='count'))
print('Total images:', len(records))

## Ablation Configs

`full_scene_rule` là pipeline đầy đủ không VLM. Các ablation bỏ từng phần scene geometry để xem ảnh hưởng. Nếu bật `RUN_VLM_RECHECK=True`, notebook thêm cấu hình `full_scene_rule_vlm_recheck` dùng Qwen đọc biển báo để xác nhận lại.

In [ ]:
def load_scene_config() -> dict:
    return json.loads(SCENE_CONFIG.read_text(encoding='utf-8'))

def write_scene_variant(name: str, drop_keys: list[str]) -> Path:
    cfg = load_scene_config()
    for key in drop_keys:
        cfg[key] = []
    path = EVAL_DIR / f'{name}.scene_config.json'
    path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
    return path

ABLATION_CONFIGS = [
    {
        'model_name': 'full_scene_rule',
        'threshold': BASE_THRESHOLD,
        'text_threshold': BASE_TEXT_THRESHOLD,
        'scene_config_path': SCENE_CONFIG,
        'use_vlm_recheck': False,
        'description': 'Detector + scene geometry + rule engine',
    },
    {
        'model_name': 'no_scene_config',
        'threshold': BASE_THRESHOLD,
        'text_threshold': BASE_TEXT_THRESHOLD,
        'scene_config_path': None,
        'use_vlm_recheck': False,
        'description': 'Remove all manually configured scene geometry',
    },
    {
        'model_name': 'no_lane_geometry',
        'threshold': BASE_THRESHOLD,
        'text_threshold': BASE_TEXT_THRESHOLD,
        'scene_config_path': write_scene_variant('no_lane_geometry', ['lanes']),
        'use_vlm_recheck': False,
        'description': 'Remove lane polygons',
    },
    {
        'model_name': 'no_no_parking_zone',
        'threshold': BASE_THRESHOLD,
        'text_threshold': BASE_TEXT_THRESHOLD,
        'scene_config_path': write_scene_variant('no_no_parking_zone', ['no_parking_zones']),
        'use_vlm_recheck': False,
        'description': 'Remove no-parking zone polygons',
    },
    {
        'model_name': 'no_stop_line',
        'threshold': BASE_THRESHOLD,
        'text_threshold': BASE_TEXT_THRESHOLD,
        'scene_config_path': write_scene_variant('no_stop_line', ['stop_lines']),
        'use_vlm_recheck': False,
        'description': 'Remove configured stop lines',
    },
]

if RUN_VLM_RECHECK:
    ABLATION_CONFIGS.append({
        'model_name': 'full_scene_rule_vlm_recheck',
        'threshold': BASE_THRESHOLD,
        'text_threshold': BASE_TEXT_THRESHOLD,
        'scene_config_path': SCENE_CONFIG,
        'use_vlm_recheck': True,
        'description': 'Full rule pipeline + Qwen sign recheck',
    })

display(pd.DataFrame(ABLATION_CONFIGS)[['model_name', 'description', 'use_vlm_recheck']])

## Prediction Helpers

In [ ]:
NO_VIOLATION = 'no_violation_or_insufficient_evidence'

def rule_prediction(result: dict) -> tuple[int, str, float]:
    candidates = [v for v in result.get('violations', []) if v.get('type') != NO_VIOLATION]
    if not candidates:
        return 0, NO_VIOLATION, 0.0
    best = max(candidates, key=lambda item: item.get('confidence', 0.0))
    return 1, best.get('type', 'unknown'), float(best.get('confidence', 0.0))

def build_sign_recheck_prompt(result: dict) -> str:
    payload = {
        'image_path': result.get('image_path'),
        'image_size': result.get('image_size'),
        'detections': result.get('detections', []),
        'facts': result.get('facts', {}),
        'rule_engine_candidates': result.get('violations', []),
    }
    return f"""
You are verifying a Vietnamese traffic violation from one image. Read visible traffic signs, symbols, lane arrows, traffic lights, and road markings. Confirm, reject, or mark uncertain for the rule-engine candidate.

Return strict JSON only:
{{"final_label":"violate | not_violate | uncertain", "final_violation_type":"illegal_parking | wrong_lane | crossing_red_light | no_violation_or_insufficient_evidence | uncertain", "confidence":0.0, "reason":"..."}}

Detector and rule output:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()

def parse_json_from_vlm(text: str) -> dict:
    import re
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?', '', cleaned).strip()
    cleaned = re.sub(r'```$', '', cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', cleaned, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def apply_vlm_recheck(vlm_reasoner, image_path: Path, result: dict, rule_label: int, rule_type: str, rule_score: float):
    prompt = build_sign_recheck_prompt(result)
    text = vlm_reasoner.reason(image_path=str(image_path), prompt=prompt)
    payload = parse_json_from_vlm(text)
    final_label = payload.get('final_label', 'uncertain')
    if final_label == 'violate':
        return 1, payload.get('final_violation_type', rule_type), float(payload.get('confidence', rule_score)), payload.get('reason', ''), text
    if final_label == 'not_violate':
        return 0, NO_VIOLATION, 1.0 - float(payload.get('confidence', 0.5)), payload.get('reason', ''), text
    return rule_label, rule_type, rule_score, payload.get('reason', ''), text

## 5-Fold Cross Validation

In [ ]:
from illegal_detector.traffic_violation_pipeline import TrafficViolationPipeline

vlm_reasoner = None
if RUN_VLM_RECHECK:
    from illegal_detector.vlm_reasoner import QwenVLReasoner
    vlm_reasoner = QwenVLReasoner(model_id=QWEN_MODEL, max_new_tokens=VLM_MAX_NEW_TOKENS)

y_all = np.array([row['y_true'] for row in records])
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
prediction_rows = []

for config in ABLATION_CONFIGS:
    print('\nRunning:', config['model_name'])
    pipeline = TrafficViolationPipeline(
        threshold=config['threshold'],
        text_threshold=config['text_threshold'],
    )

    for fold, (_, val_idx) in enumerate(skf.split(np.zeros(len(records)), y_all), start=1):
        fold_records = [records[i] for i in val_idx]
        print(f'  fold {fold}/{N_SPLITS}: {len(fold_records)} images')

        for item in fold_records:
            start = time.perf_counter()
            row = {
                'model_name': config['model_name'],
                'fold': fold,
                'image_path': item['relative_path'],
                'task': item['task'],
                'y_true': item['y_true'],
                'status': 'ok',
                'error': '',
            }
            try:
                result = pipeline.analyze(
                    image_path=str(item['image_path']),
                    scene_config_path=str(config['scene_config_path']) if config['scene_config_path'] else None,
                    annotate_path=None,
                    segment_dir=None,
                    run_vlm=False,
                )
                y_pred, pred_type, score = rule_prediction(result)
                vlm_reason = ''
                if config['use_vlm_recheck']:
                    y_pred, pred_type, score, vlm_reason, _ = apply_vlm_recheck(
                        vlm_reasoner, item['image_path'], result, y_pred, pred_type, score
                    )
                row.update({
                    'y_pred': y_pred,
                    'score': score,
                    'predicted_type': pred_type,
                    'num_detections': len(result.get('detections', [])),
                    'num_violations': len(result.get('violations', [])),
                    'vlm_reason': vlm_reason,
                })
            except Exception as exc:
                row.update({
                    'y_pred': -1,
                    'score': 0.0,
                    'predicted_type': 'error',
                    'num_detections': 0,
                    'num_violations': 0,
                    'vlm_reason': '',
                    'status': 'error',
                    'error': ''.join(traceback.format_exception_only(type(exc), exc)).strip(),
                })
            row['predict_time_sec'] = time.perf_counter() - start
            prediction_rows.append(row)

pred_df = pd.DataFrame(prediction_rows)
pred_path = EVAL_DIR / 'cv_predictions.csv'
pred_df.to_csv(pred_path, index=False)
print('Saved predictions:', pred_path)
display(pred_df.head())

## Metrics: F1/Recall/Precision Micro, Macro, Weighted + ROC-AUC + PR-AUC

In [ ]:
def compute_metrics(group: pd.DataFrame) -> dict:
    ok = group[group['status'] == 'ok'].copy()
    y_true = ok['y_true'].astype(int).to_numpy()
    y_pred = ok['y_pred'].astype(int).to_numpy()
    scores = ok['score'].astype(float).to_numpy()

    output = {
        'n': len(ok),
        'errors': int((group['status'] != 'ok').sum()),
        'accuracy': accuracy_score(y_true, y_pred) if len(ok) else 0.0,
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0) if len(ok) else 0.0,
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0) if len(ok) else 0.0,
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0) if len(ok) else 0.0,
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0) if len(ok) else 0.0,
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0) if len(ok) else 0.0,
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0) if len(ok) else 0.0,
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0) if len(ok) else 0.0,
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0) if len(ok) else 0.0,
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0) if len(ok) else 0.0,
        'mean_predict_time_sec': ok['predict_time_sec'].mean() if len(ok) else 0.0,
        'total_predict_time_sec': ok['predict_time_sec'].sum() if len(ok) else 0.0,
    }
    if len(np.unique(y_true)) == 2:
        output['roc_auc'] = roc_auc_score(y_true, scores)
        output['pr_auc'] = average_precision_score(y_true, scores)
    else:
        output['roc_auc'] = np.nan
        output['pr_auc'] = np.nan
    return output

metric_rows = []
for (model_name, fold), group in pred_df.groupby(['model_name', 'fold']):
    row = {'model_name': model_name, 'fold': fold}
    row.update(compute_metrics(group))
    metric_rows.append(row)

fold_metrics_df = pd.DataFrame(metric_rows)
summary_df = fold_metrics_df.groupby('model_name').agg(['mean', 'std']).reset_index()
flat_cols = []
for col in summary_df.columns:
    flat_cols.append(col[0] if col[1] == '' else f'{col[0]}_{col[1]}')
summary_df.columns = flat_cols

fold_metrics_df.to_csv(EVAL_DIR / 'fold_metrics.csv', index=False)
summary_df.to_csv(EVAL_DIR / 'summary_metrics.csv', index=False)

display(summary_df.sort_values('f1_weighted_mean', ascending=False))

## Main Results: Tables And Charts

In [ ]:
main_cols = [
    'model_name', 'accuracy_mean', 'f1_weighted_mean', 'f1_macro_mean',
    'recall_weighted_mean', 'recall_macro_mean', 'roc_auc_mean', 'pr_auc_mean',
    'mean_predict_time_sec_mean',
]
main_results = summary_df[main_cols].sort_values('f1_weighted_mean', ascending=False)
display(main_results)

plot_df = main_results.set_index('model_name')[['f1_weighted_mean', 'f1_macro_mean', 'recall_weighted_mean', 'roc_auc_mean', 'pr_auc_mean']]
ax = plot_df.plot(kind='bar', figsize=(12, 5), ylim=(0, 1), title='Main evaluation metrics by model/config')
ax.set_ylabel('Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(EVAL_DIR / 'main_results_bar.png', dpi=200)
plt.show()

## ROC And PR Curves

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for model_name, group in pred_df[pred_df['status'] == 'ok'].groupby('model_name'):
    y_true = group['y_true'].astype(int).to_numpy()
    scores = group['score'].astype(float).to_numpy()
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, scores)
        plt.plot(fpr, tpr, label=model_name)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.title('ROC curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
for model_name, group in pred_df[pred_df['status'] == 'ok'].groupby('model_name'):
    y_true = group['y_true'].astype(int).to_numpy()
    scores = group['score'].astype(float).to_numpy()
    if len(np.unique(y_true)) == 2:
        precision, recall, _ = precision_recall_curve(y_true, scores)
        plt.plot(recall, precision, label=model_name)
plt.title('Precision-Recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'roc_pr_curves.png', dpi=200)
plt.show()

## Ablation Study

Đọc bảng này theo hướng: nếu bỏ module mà F1/Recall/AUC giảm mạnh, module đó đang đóng góp nhiều. Nếu bỏ module mà metric tăng, module đó có thể đang gây false positive hoặc cấu hình chưa phù hợp.

In [ ]:
base_name = 'full_scene_rule'
base = main_results[main_results['model_name'] == base_name].iloc[0]
ablation_rows = []
for _, row in main_results.iterrows():
    item = row.to_dict()
    item['delta_f1_weighted_vs_full'] = row['f1_weighted_mean'] - base['f1_weighted_mean']
    item['delta_recall_weighted_vs_full'] = row['recall_weighted_mean'] - base['recall_weighted_mean']
    item['delta_roc_auc_vs_full'] = row['roc_auc_mean'] - base['roc_auc_mean']
    ablation_rows.append(item)

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df[['model_name', 'f1_weighted_mean', 'delta_f1_weighted_vs_full', 'recall_weighted_mean', 'delta_recall_weighted_vs_full', 'roc_auc_mean', 'delta_roc_auc_vs_full']])
ablation_df.to_csv(EVAL_DIR / 'ablation_study.csv', index=False)

## Error Analysis / Case Study

Chọn các case sai để viết phân tích định tính: ảnh bị phân loại sai, dự đoán sai kiểu gì, khả năng do detector, scene config, biển báo không đọc được, hoặc luật single-image chưa đủ chứng cứ.

In [ ]:
from IPython.display import display
from PIL import Image

best_model = main_results.iloc[0]['model_name']
errors = pred_df[(pred_df['model_name'] == best_model) & (pred_df['status'] == 'ok') & (pred_df['y_true'] != pred_df['y_pred'])].copy()
errors = errors.sort_values('score', ascending=False)
display(errors[['image_path', 'task', 'y_true', 'y_pred', 'score', 'predicted_type', 'num_detections', 'num_violations', 'vlm_reason']].head(10))

for _, row in errors.head(5).iterrows():
    print('\nCase:', row['image_path'])
    print('true=', row['y_true'], 'pred=', row['y_pred'], 'score=', row['score'], 'type=', row['predicted_type'])
    print('Suggested analysis: kiểm tra biển báo có đọc được không, vùng cấm/lane polygon có quá rộng không, detector có nhầm object không, ảnh đơn có đủ bằng chứng không.')
    display(Image.open(PROJECT_ROOT / row['image_path']))

errors.to_csv(EVAL_DIR / 'error_cases.csv', index=False)

## Timing Sentence

In [ ]:
best_timing = main_results.iloc[0]
sentence = (
    f"Training time is not applicable because the system uses pre-trained GroundingDINO/Qwen and a rule engine; "
    f"on the evaluation set, the best configuration ({best_timing['model_name']}) required "
    f"{best_timing['mean_predict_time_sec_mean']:.2f} seconds per image on average for prediction."
)
print(sentence)
(EVAL_DIR / 'timing_sentence.txt').write_text(sentence, encoding='utf-8')

## Previous Studies Comparison Template

Repo hiện tại không chứa baseline từ nghiên cứu trước trên cùng dataset. Nếu có paper/model cùng dataset, điền thêm vào bảng dưới. Không nên tự bịa số liệu; chỉ đưa kết quả đã chạy hoặc đã trích dẫn rõ nguồn.

In [ ]:
previous_studies = pd.DataFrame([
    {
        'study_or_model': 'This work - best config',
        'dataset': 'current project dataset',
        'f1_weighted': float(best_timing['f1_weighted_mean']),
        'roc_auc': float(best_timing['roc_auc_mean']),
        'notes': 'Pre-trained detector + rule/VLM pipeline evaluated by 5-fold CV',
    },
    {
        'study_or_model': 'Previous study name here',
        'dataset': 'same dataset or external dataset name',
        'f1_weighted': None,
        'roc_auc': None,
        'notes': 'Fill only with cited/reproduced results',
    },
])
display(previous_studies)
previous_studies.to_csv(EVAL_DIR / 'previous_studies_comparison_template.csv', index=False)

## Discussion Notes

Gợi ý viết Discussion:

- Nếu `full_scene_rule` tốt hơn `no_scene_config`, kết quả chứng minh scene geometry giúp giảm ambiguity của ảnh đơn.
- Nếu `no_no_parking_zone` làm recall giảm, no-parking zone là module quan trọng cho illegal parking.
- Nếu false positive cao ở `not_violate`, gap chính là detector/luật chưa hiểu ý nghĩa biển báo hoặc chưa có temporal evidence.
- Nếu bật VLM recheck và false positive giảm, kết quả trả lời RQ rằng VLM hữu ích ở vai trò đọc biển báo/xác nhận ngữ nghĩa thay vì thay detector.
- Liên hệ Introduction gap: object detection chỉ thấy object, nhưng không hiểu luật/biển báo; pipeline kết hợp geometry + VLM để lấp gap đó.